In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS silver;

In [0]:
%sql

CREATE OR REPLACE TABLE silver.calidad_aire AS

SELECT
    CAST(fecha_hora AS TIMESTAMP) AS fecha_hora,
    municipio,
    CAST(latitud AS DOUBLE) AS latitud,
    CAST(longitud AS DOUBLE) AS longitud,
    CAST(pm10 AS DOUBLE) AS pm10,
    CAST(pm2_5 AS DOUBLE) AS pm2_5,
    CAST(us_aqi AS DOUBLE) AS us_aqi

FROM bronze.calidad_aire

WHERE municipio IS NOT NULL;

In [0]:
%sql

SELECT *
FROM silver.calidad_aire
LIMIT 20;

In [0]:
%sql

CREATE OR REPLACE TABLE silver.hospitalizaciones AS

SELECT
    h.id_hospitalizacion,
    h.id_paciente,
    h.id_medico,
    h.fecha_ingreso,
    h.fecha_alta
FROM bronze.hospitalizacion h
WHERE h.id_paciente IS NOT NULL;

In [0]:
%sql

SELECT
    id_hospitalizacion,
    id_paciente,
    id_medico,
    fecha_ingreso,
    fecha_alta,
    motivo
FROM bronze.hospitalizacion
LIMIT 20;

In [0]:
%sql

CREATE OR REPLACE TABLE silver.hospitalizaciones AS

SELECT
    CAST(id_hospitalizacion AS INT) AS id_hospitalizacion,
    CAST(id_paciente AS INT) AS id_paciente,
    CAST(id_medico AS INT) AS id_medico,
    CAST(fecha_ingreso AS DATE) AS fecha_ingreso,
    CAST(fecha_alta AS DATE) AS fecha_alta,
    motivo
FROM bronze.hospitalizacion
WHERE id_paciente IS NOT NULL
  AND fecha_ingreso IS NOT NULL
  AND fecha_alta IS NOT NULL;

In [0]:
%sql

SELECT *
FROM silver.hospitalizaciones
LIMIT 20;

In [0]:
%sql

SELECT COUNT(*) AS total_registros
FROM silver.hospitalizaciones;

In [0]:
%sql

CREATE OR REPLACE TABLE silver.pacientes AS

SELECT
    CAST(id_paciente AS INT) AS id_paciente,
    nombre,
    apellido
FROM bronze.paciente
WHERE id_paciente IS NOT NULL;

In [0]:
%sql

SELECT *
FROM silver.pacientes
LIMIT 20;

In [0]:
%sql

SELECT COUNT(*) AS total_registros
FROM silver.pacientes;

In [0]:
%sql

CREATE OR REPLACE TABLE silver.diagnosticos AS

SELECT
    CAST(id_diagnostico AS INT) AS id_diagnostico,
    nombre,
    descripcion
FROM bronze.diagnostico
WHERE id_diagnostico IS NOT NULL;

In [0]:
%sql

SELECT *
FROM silver.diagnosticos
LIMIT 20;

In [0]:
%sql

SELECT COUNT(*) AS total_registros
FROM silver.diagnosticos;

In [0]:
%sql

CREATE OR REPLACE TABLE silver.recetas AS

SELECT
    CAST(id_receta AS INT) AS id_receta,
    CAST(id_paciente AS INT) AS id_paciente,
    CAST(id_medico AS INT) AS id_medico,
    CAST(fecha_receta AS DATE) AS fecha_receta
FROM bronze.receta
WHERE id_receta IS NOT NULL
  AND id_paciente IS NOT NULL
  AND fecha_receta IS NOT NULL;

In [0]:
%sql

SELECT *
FROM silver.recetas
LIMIT 20;

In [0]:
%sql

SELECT COUNT(*) AS total_registros
FROM silver.recetas;

In [0]:
%sql

CREATE OR REPLACE TABLE silver.detalle_receta AS

SELECT
    CAST(id_receta AS INT) AS id_receta,
    CAST(id_medicamento AS INT) AS id_medicamento,
    CAST(cantidad AS INT) AS cantidad
FROM bronze.detalle_receta
WHERE id_receta IS NOT NULL
  AND id_medicamento IS NOT NULL;

In [0]:
%sql

SELECT COUNT(*) AS total_registros
FROM silver.detalle_receta;

In [0]:
%sql

CREATE OR REPLACE TABLE silver.medicamentos AS

SELECT
    CAST(id_medicamento AS INT) AS id_medicamento,
    nombre,
    fabricante
FROM bronze.medicamento
WHERE id_medicamento IS NOT NULL;

In [0]:
%sql

SELECT COUNT(*) AS total_registros
FROM silver.medicamentos;

In [0]:
%sql

CREATE OR REPLACE TABLE silver.historial_clinico AS

SELECT
    h.id_hospitalizacion,
    h.id_paciente,
    p.nombre,
    p.apellido,
    h.id_medico,
    h.fecha_ingreso,
    h.fecha_alta,
    h.motivo
FROM silver.hospitalizaciones h
LEFT JOIN silver.pacientes p
    ON h.id_paciente = p.id_paciente;

In [0]:
%sql

SELECT *
FROM silver.historial_clinico
LIMIT 20;


In [0]:
%sql

SELECT COUNT(*) AS total_registros
FROM silver.historial_clinico;

In [0]:
%sql

SELECT COUNT(*) AS registros
FROM silver.hospitalizaciones;

In [0]:
%sql
CREATE OR REPLACE TABLE silver.medicamento_calidad_aire AS

SELECT 
    m.id_medicamento,
    m.nombre,
    m.tipo,
    m.fabricante,
    m.stock,
    dr.id_receta,
    dr.dosis,
    dr.frecuencia,
    dr.duracion_dias,
    dr.cantidad,
    r.id_paciente,
    r.fecha_receta,
    p.nombre AS nombre_paciente,
    p.direccion AS municipio_paciente,
    c.fecha_hora,
    c.pm10,
    c.pm2_5,
    c.us_aqi,
    c.municipio AS municipio_api

FROM bronze.medicamento m
JOIN bronze.detalle_receta dr
    ON m.id_medicamento = dr.id_medicamento
JOIN bronze.receta r
    ON dr.id_receta = r.id_receta
JOIN bronze.paciente p
    ON r.id_paciente = p.id_paciente
JOIN bronze.calidad_aire c
    ON LOWER(TRIM(p.direccion)) = LOWER(TRIM(c.municipio));

In [0]:
%sql
SELECT *
FROM silver.medicamento_calidad_aire
LIMIT 20;

In [0]:
# Prueba de conexión con Git - Silver
print("✅ Silver está conectado a Git correctamente")
print("Prueba de Commit & Push")